In [2]:
#匯入套件與設定環境
try:
  import ultralytics
  print("Ultralytics 已安裝，版本:", ultralytics.__version__)
except ImportError:
  print("Ultralytics 未安裝，正在安裝...")
  !pip install ultralytics
import numpy as np
import matplotlib.pyplot as plt
import torch
import os
import requests
import shutil
import yaml
import re
from PIL import Image
from PIL import ImageDraw
from ultralytics import YOLO
from math import ceil

Ultralytics 已安裝，版本: 8.3.97


In [3]:
#在Colab建立資料夾
colab_save_path = "/content"

folder_name = "Where is Waldo.v3i.yolov8"
folder_name_split = "Where is Waldo.v3i.yolov8_split"

folder_list = ["train", "valid", "test"]
folder_list_2 = ["images", "labels"]

for folder in folder_list:
  for folder_2 in folder_list_2:
    folder_path = os.path.join(folder_name_split, folder, folder_2)
    if not os.path.exists(folder_path):
      os.makedirs(folder_path)
      print(f"建立資料夾: {folder_path}")

建立資料夾: Where is Waldo.v3i.yolov8_split/train/images
建立資料夾: Where is Waldo.v3i.yolov8_split/train/labels
建立資料夾: Where is Waldo.v3i.yolov8_split/valid/images
建立資料夾: Where is Waldo.v3i.yolov8_split/valid/labels
建立資料夾: Where is Waldo.v3i.yolov8_split/test/images
建立資料夾: Where is Waldo.v3i.yolov8_split/test/labels


In [4]:
#讀取檔案並解壓縮
zip_url = "https://raw.githubusercontent.com/Zong0120/Where_is_Waldo/main/Where%20is%20Waldo.v3i.yolov8.zip"
zip_name = "Where is Waldo.v3i.yolov8.zip"

response = requests.get(zip_url, stream=True)  #以串流方式下載
with open(zip_name, "wb") as file:
    for chunk in response.iter_content(chunk_size=8192):  #以8KB為單位寫入檔案
        file.write(chunk)
print(f"下載成功：{zip_name}")

#解壓縮
shutil.unpack_archive(zip_name, folder_name)
print(f"解壓縮完成：{folder_name}")

下載成功：Where is Waldo.v3i.yolov8.zip
解壓縮完成：Where is Waldo.v3i.yolov8


In [5]:
#取得資料夾所有符合附檔名的檔案
def get_files_path(folder_path, file_extension):

  all_files = os.listdir(folder_path)

  files = [f for f in all_files if any(f.endswith(ext) for ext in file_extension)]

  files_path = [os.path.join(folder_path, single_file) for single_file in files]

  return files_path

In [6]:
#修改檔名
def extract_number(file_path):
  file_name = os.path.basename(file_path)
  #抓出開頭的數字
  match = re.match(r"(\d+)_jpg", file_name)
  if match:
    return int(match.group(1))
  else:
    return float("inf")

for i in range(len(folder_list)):
  for j in range(len(folder_list_2)):
    folder_path = os.path.join(colab_save_path, folder_name, folder_list[i], folder_list_2[j])

    ext = ".jpg" if folder_list_2[j] == "images" else ".txt"
    all_file_paths = get_files_path(folder_path, ext)

    for file_path in all_file_paths:
      num = extract_number(file_path)

      if num < 10:
        new_name = f"{num:02d}{ext}"
      else:
        new_name = f"{num}{ext}"

      old_path = file_path
      new_path = os.path.join(folder_path, new_name)
      os.rename(old_path, new_path)

print("檔名修改成功")

檔名修改成功


In [7]:
#複製並修改yaml
source_yaml = os.path.join(colab_save_path, folder_name, "data.yaml")
dest_yaml = os.path.join(colab_save_path, folder_name_split, "data.yaml")

shutil.copy(source_yaml, dest_yaml)

#讀取
with open(dest_yaml, "r") as file:
    data = yaml.safe_load(file)

#修改
str_list = ["train", "val", "test"]

for i in range(len(str_list)):
  data[f"{str_list[i]}"] = f"{folder_list[i]}/{folder_list_2[0]}"

#寫入
with open(dest_yaml, "w") as file:
    yaml.safe_dump(data, file, default_flow_style=False)

print("yaml修改完成")
!cat "{dest_yaml}"

yaml修改完成
names:
- Waldo
nc: 1
roboflow:
  license: CC BY 4.0
  project: where-is-waldo-pimke
  url: https://universe.roboflow.com/geass987654/where-is-waldo-pimke/dataset/3
  version: 3
  workspace: geass987654
test: test/images
train: train/images
val: valid/images


In [8]:
#分割圖片
def generate_image_block(image, labels, block_size, overlap_ratio):

  width, height = image.size
  stride = int(block_size[0] * (1 - overlap_ratio))
  num_row = ceil((height - block_size[1]) / stride) + 1
  num_col = ceil((width - block_size[0]) / stride) + 1

  label_list = []

  for label_str in labels:
    label = label_str.split(" ")

    #還原bounding box在原圖的座標
    label_xy = (float(label[1]) * width, float(label[2]) * height)
    label_wh = (float(label[3]) * width, float(label[4]) * height)
    label_id = label[0]

    label_list.append((label_xy, label_wh, label_id))

  for i in range(num_row):
    block_y1 = i * stride

    #最後一列
    if block_y1 + block_size[1] > height:
      block_y1 = height - block_size[1]

    for j in range(num_col):
      block_x1 = j * stride

      #最後一行
      if block_x1 + block_size[0] > width:
        block_x1 = width - block_size[0]

      block_x2 = block_x1 + block_size[0]
      block_y2 = block_y1 + block_size[1]

      block = image.crop((block_x1, block_y1, block_x2, block_y2))

      label_text = ""

      #調整label
      for label_xy, label_wh, label_id in label_list:

        block_xy = (block_x1, block_y1)
        block_wh = (block_size[0], block_size[1])

        label_final = get_label_in_block(block_xy, block_wh, label_xy, label_wh, label_id, 0.2)
        # print(f"區塊 ({i}, {j}) 的標籤: {label_final}")
        if len(label_final) > 0:
          label_text += label_final + "\n"
      #儲存到Colab
      file_name = f"{img_name}_{i:02}_{j:02}"
      image_path = os.path.join(colab_save_path, folder_name_split, set_name, "images", f"{file_name}.jpg")
      label_path = os.path.join(colab_save_path, folder_name_split, set_name, "labels", f"{file_name}.txt")

      block.save(image_path)

      with open(label_path, "w") as f:
        f.write(label_text)

In [9]:
#轉換bounding box座標
def get_label_in_block(block_xy, block_wh, label_xy, label_wh, label_id, min_visible_ratio):

  block_x , block_y = block_xy  #區塊在原圖的座標
  block_w , block_h = block_wh  #區塊寬高
  label_x , label_y = label_xy  #bounding box的中心在原圖的座標
  label_w , label_h = label_wh  #bounding box的寬高

  #左上角和右下角
  x1 = label_x - label_w / 2
  y1 = label_y - label_h / 2
  x2 = label_x + label_w / 2
  y2 = label_y + label_h / 2

  #bounding box在區塊內的座標(不超出區塊邊界)
  clipped_x1 = max(x1, block_x)
  clipped_y1 = max(y1, block_y)
  clipped_x2 = min(x2, block_x + block_w)
  clipped_y2 = min(y2, block_y + block_h)

  #bounding box面積和分割後的面積
  area = label_w * label_h
  clipped_w = max(0, clipped_x2 - clipped_x1)
  clipped_h = max(0, clipped_y2 - clipped_y1)
  clipped_area = clipped_w * clipped_h

  #原圖不存在威利，label為空

  #bounding box在區塊邊界上
  if clipped_area == 0:
    return ""

  #比例達到標準，保留bounding box
  visible_ratio = clipped_area / area

  if visible_ratio < min_visible_ratio:
    return ""

  #新bounding box的中心點(平移到原圖的左上角區塊)
  new_label_x = (clipped_x1 + clipped_x2) / 2 - block_x
  new_label_y = (clipped_y1 + clipped_y2) / 2 - block_y

  #縮放到區塊大小，介於0-1
  norm_new_label_x = round(new_label_x / block_w, 6)
  norm_new_label_y = round(new_label_y / block_h, 6)
  norm_clipped_w = round(clipped_w / block_w, 6)
  norm_clipped_h = round(clipped_h / block_h, 6)

  label_str = f"{label_id} {norm_new_label_x} {norm_new_label_y} {norm_clipped_w} {norm_clipped_h}"

  return label_str

In [15]:
# #顯示資料夾
# !ls "{colab_save_path}"

# #顯示原圖資料夾的images
# !ls "{colab_save_path}/{folder_name}/{folder_list[0]}/{folder_list_2[0]}"

# #顯示分割圖片資料夾的images
# !ls "{colab_save_path}/{folder_name_split}/{folder_list[2]}/{folder_list_2[0]}"

#顯示預測的txt
!ls "runs/detect/predict/labels"


# #刪除原圖資料夾
# !rm -r "{colab_save_path}/{folder_name}"

# #刪除分割資料夾
# !rm -r "{colab_save_path}/{folder_name_split}"

12_00_02.txt  12_01_02.txt  12_02_02.txt  12_03_02.txt	14_09_00.txt  14_10_01.txt
12_00_03.txt  12_01_03.txt  12_02_03.txt  12_03_03.txt	14_09_01.txt  14_11_00.txt
12_00_04.txt  12_01_04.txt  12_02_04.txt  12_03_04.txt	14_10_00.txt  14_11_01.txt


In [10]:
#圖片前處理
block_size = (640, 640)
overlap_ratio = 0.75

for i in range(len(folder_list)):
  print(f"處理資料夾: {folder_list[i]}")
  image_folder_path = os.path.join(colab_save_path, folder_name, folder_list[i], folder_list_2[0])
  label_folder_path = os.path.join(colab_save_path, folder_name, folder_list[i], folder_list_2[1])
  set_name = folder_list[i]
  image_paths = get_files_path(image_folder_path, ".jpg")

  for img_path in image_paths:
    #打開圖片
    image = Image.open(img_path)
    img_name = os.path.basename(img_path)
    img_name = os.path.splitext(img_name)[0]

    #打開label
    label_path = os.path.join(label_folder_path, f"{img_name}.txt")

    #轉換成label list(去掉頭尾的空白和換行字元)
    with open(label_path, "r") as f:
      labels = [line.strip() for line in f]

    #分割原圖
    generate_image_block(image, labels, block_size, overlap_ratio)
    print(f"{img_name} 分割完成")

處理資料夾: train
16 分割完成
02 分割完成
01 分割完成
11 分割完成
13 分割完成
15 分割完成
06 分割完成
19 分割完成
09 分割完成
04 分割完成
03 分割完成
05 分割完成
18 分割完成
10 分割完成
17 分割完成
處理資料夾: valid
07 分割完成
08 分割完成
處理資料夾: test
14 分割完成
12 分割完成


In [ ]:
#訓練模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = os.path.join(colab_save_path, folder_name_split, "data.yaml")

model = YOLO("yolov8n.pt").to(device)
result_train = model.train(data=data_path, epochs=100, imgsz=640, batch=16, device=device)

engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/Where is Waldo.v3i.yolov8_split/data.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=cuda, workers=8, project=None, name=train3, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_width=None, format=torchscript, keras=False,

100%|██████████| 755k/755k [00:00<00:00, 20.1MB/s]


Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

100%|██████████| 5.35M/5.35M [00:00<00:00, 102MB/s]


AMP: checks passed ✅


...

    100/100      4.24G     0.4261     0.3249     0.8013          7        640: 100%|██████████| 45/45 [00:11<00:00,  3.80it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:01<00:00,  5.73it/s]

                   all        180         33      0.276      0.485      0.319      0.183



100 epochs completed in 0.408 hours.
Optimizer stripped from runs/detect/train3/weights/last.pt, 6.3MB
Optimizer stripped from runs/detect/train3/weights/best.pt, 6.3MB

Validating runs/detect/train3/weights/best.pt...
Ultralytics 8.3.96 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:02<00:00,  2.47it/s]


                   all        180         33      0.967      0.485      0.499      0.259
Speed: 0.4ms preprocess, 2.4ms inference, 0.0ms loss, 4.6ms postprocess per image
Results saved to runs/detect/train3


In [ ]:
#驗證模型
result_val = model.val(data=data_path, epochs=100, imgsz=640, batch=16, device=device)

Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning /content/Where is Waldo.v3i.yolov8_split/valid/labels.cache... 180 images, 149 backgrounds, 0 corrupt: 100%|██████████| 180/180 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:03<00:00,  3.46it/s]


                   all        180         33      0.967      0.485      0.495      0.262
Speed: 0.3ms preprocess, 10.1ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to runs/detect/train32


In [11]:
#掛載Google Drive並建立資料夾
from google.colab import drive

drive.mount('/content/drive')
drive_save_path = "/content/drive/MyDrive/WheresWaldo_with_YOLOv8"

if not os.path.exists(drive_save_path):
  os.makedirs(drive_save_path)
  print(f"建立資料夾: {drive_save_path}")

Mounted at /content/drive


In [ ]:
#保存模型
from datetime import datetime
from google.colab import drive

#取得當前時間作為字串
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

model_path = os.path.join(drive_save_path, "saved_models")
last_model = os.path.join(colab_save_path, "runs", "detect", "train3", "weights", "last.pt")
best_model = os.path.join(colab_save_path, "runs", "detect", "train3", "weights", "best.pt")

if not os.path.exists(model_path):
  os.makedirs(model_path)
  print(f"建立資料夾: {model_path}")


shutil.copy(last_model, os.path.join(model_path, f"{timestamp}_last.pt"))
shutil.copy(best_model, os.path.join(model_path, "best.pt"))
print(f"保存完畢")

保存完畢


In [12]:
#預測模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = os.path.join(drive_save_path, "saved_models")

best_model_path = os.path.join(model_path, "best.pt")
source_path = os.path.join(colab_save_path, folder_name_split, "test", "images")
result_path = os.path.join(colab_save_path, "runs", "detect", "predict")

model = YOLO(best_model_path).to(device)
result_test = model.predict(source_path, conf=0.7, save=True, save_txt=True, save_conf=True, device=device)


image 1/136 /content/Where is Waldo.v3i.yolov8_split/test/images/12_00_00.jpg: 640x640 (no detections), 445.3ms
image 2/136 /content/Where is Waldo.v3i.yolov8_split/test/images/12_00_01.jpg: 640x640 (no detections), 223.1ms
image 3/136 /content/Where is Waldo.v3i.yolov8_split/test/images/12_00_02.jpg: 640x640 1 Waldo, 225.5ms
image 4/136 /content/Where is Waldo.v3i.yolov8_split/test/images/12_00_03.jpg: 640x640 1 Waldo, 203.1ms
image 5/136 /content/Where is Waldo.v3i.yolov8_split/test/images/12_00_04.jpg: 640x640 1 Waldo, 214.9ms
image 6/136 /content/Where is Waldo.v3i.yolov8_split/test/images/12_01_00.jpg: 640x640 (no detections), 206.3ms
image 7/136 /content/Where is Waldo.v3i.yolov8_split/test/images/12_01_01.jpg: 640x640 (no detections), 206.2ms
image 8/136 /content/Where is Waldo.v3i.yolov8_split/test/images/12_01_02.jpg: 640x640 1 Waldo, 216.9ms
image 9/136 /content/Where is Waldo.v3i.yolov8_split/test/images/12_01_03.jpg: 640x640 1 Waldo, 204.3ms
image 10/136 /content/Where is 

In [22]:
#測試
result_test.sort(key=lambda x: x.path)

for result in result_test:
  if len(result_test) > 0:
    print(f"result[{i}] name:{result_test[i].path.split('/')[-1]} location:{result_test[i].boxes.xyxy}\n")
    break
all_label_paths = get_files_path("runs/detect/predict/labels", ".txt")
all_label_paths.sort()
with open(all_label_paths[0], "r") as f:
  labels = [line.strip() for line in f]

  for label_str in labels:
    label = label_str.split(" ")
    x = round(float(label[1]) * 640)
    y = round(float(label[2]) * 640)
    w = round(float(label[3]) * 640)
    h = round(float(label[4]) * 640)

    x1 = x - w // 2
    y1 = y - h // 2
    x2 = x + w // 2
    y2 = y + h // 2

    print(f"label[0] name:{all_label_paths[0].split('/')[-1]}, location:{(x1, y1), (x2, y2)}")
    break

result[2] name:12_00_02.jpg location:tensor([[524.5286, 523.0630, 558.8845, 569.7178]])

label[0] name:12_00_02.txt, location:((525, 523), (559, 569))


In [ ]:
#合併在區塊中被分割的邊界框
def merge_bounding_box(detections, threshold=30):

  merged_boxes = []

  #先按y排序，再按x排序
  detections.sort(key=lambda b: (b[1], b[0]))

  while detections:
    class_id, x1, y1, x2, y2, confidence = detections.pop(0)
    merged = False

    for i, (m_class_id, mx1, my1, mx2, my2, m_confidence) in enumerate(merged_boxes):
      #合併相同類別的框
      if class_id == m_class_id:
        #水平和垂直距離
        dx = min(abs(mx2 - x1), abs(x2 - mx1))
        dy = min(abs(my2 - y1), abs(y2 - my1))

        if dx < threshold and dy < threshold:
          #更新框的範圍
          merged_boxes[i] = [
              class_id,
              min(mx1, x1), min(my1, y1),
              max(mx2, x2), max(my2, y2),
              (m_confidence + confidence) / 2
          ]
          merged = True
          break

    if not merged:
        merged_boxes.append([class_id, x1, y1, x2, y2, confidence])

  return merged_boxes

In [ ]:
import os
import cv2
import re
import numpy as np
import matplotlib.pyplot as plt
from math import ceil

def draw_predictions(label_folder, image_folder, class_names, block_size=640, overlap_ratio=0.75):
    stride = int(block_size * (1 - overlap_ratio))

    all_label_paths = sorted([os.path.join(label_folder, f) for f in os.listdir(label_folder) if f.endswith(".txt")])
    all_image_paths = sorted([os.path.join(image_folder, f) for f in os.listdir(image_folder) if f.endswith(".jpg")])

    for image_path in all_image_paths:
        img_name = os.path.basename(image_path).split("_")[0]
        original_img = cv2.imread(image_path)
        height, width = original_img.shape[:2]

        num_row = ceil((height - block_size) / stride) + 1
        num_col = ceil((width - block_size) / stride) + 1

        detections = []

        for label_path in all_label_paths:
            label_name = os.path.basename(label_path)
            match = re.search(r"(\d+)_(\d+)_(\d+)\.txt", label_name)
            if not match:
                continue

            row, col = int(match.group(1)), int(match.group(2))
            if row + 1 == num_row:
                offset_y = height - block_size
            else:
                offset_y = row * stride

            if col + 1 == num_col:
                offset_x = width - block_size
            else:
                offset_x = col * stride

            with open(label_path, "r") as f:
                labels = [line.strip() for line in f]

            for label_str in labels:
                parts = label_str.split(" ")
                class_id = int(parts[0])
                x = int(float(parts[1]) * block_size)
                y = int(float(parts[2]) * block_size)
                w = int(float(parts[3]) * block_size)
                h = int(float(parts[4]) * block_size)
                conf = float(parts[5])

                x1 = x - w // 2 + offset_x
                y1 = y - h // 2 + offset_y
                x2 = x + w // 2 + offset_x
                y2 = y + h // 2 + offset_y

                detections.append((class_id, x1, y1, x2, y2, conf))

        # Merge bounding boxes (if needed)
        merged_boxes = merge_bounding_box(detections)

        # Draw results
        for class_id, x1, y1, x2, y2, confidence in merged_boxes:
            label = f"{class_names[class_id]} {confidence:.2f}"
            cv2.rectangle(original_img, (x1, y1), (x2, y2), (0, 0, 255), 2)
            (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(original_img, (x1, y1 - h - 5), (x1 + w, y1), (0, 0, 255), -1)
            cv2.putText(original_img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10, 10))
        plt.imshow(original_img)
        plt.axis("off")
        plt.show()

In [ ]:
import shutil
from google.colab import files
label_folder = "runs/detect/predict2/labels"
image_folder = os.path.join(colab_save_path, folder_name, folder_list[2], "images")


# draw_predictions(label_folder, image_folder, result_test[0].names)

FileNotFoundError: Cannot find file: 12_01_03.txt

In [ ]:
#顯示預測結果
import cv2
import re

pred_folder_path = "runs/detect/predict2/labels"
image_folder_path = os.path.join(colab_save_path, folder_name_split, folder_list[2], "images")
original_img_path = os.path.join(colab_save_path, folder_name, folder_list[2], "images")

all_predict_paths = get_files_path(pred_folder_path, ".txt")
all_predict_paths.sort()
all_original_img_paths = get_files_path(original_img_path, ".jpg")
all_original_img_paths.sort()

block_size = 640
overlap_ratio = 0.75
stride = int(block_size * (1 - overlap_ratio))

#所有類別名稱
class_name = result_test[0].names

last_name = all_predict_paths[0].split("/")[-1].split("_")[0]
current_name = ""

for original_img_path in all_original_img_paths:
  img_name = os.path.basename(original_img_path).split("_")[0]
  original_img = cv2.imread(original_img_path).copy()

  height, width = original_img.shape[:2]
  num_row = ceil((height - block_size) / stride) + 1
  num_col = ceil((width - block_size) / stride) + 1

  detection = []

  for predict_path in all_predict_paths:
    img_name = os.path.basename(predict_path)
    current_name = img_name.split("_")[0]

    if current_name != last_name:
      last_name = current_name

      merged_boxes = merge_bounding_box(detection)

      #繪製邊界框跟label(紅底白字)
      for class_id, x1, y1, x2, y2, confidence in merged_boxes:
        label = f"{class_name[class_id]} {confidence:.2f}"
        # print(f"label:{label}, location:{(x1, y1), (x2, y2)}")
        cv2.rectangle(original_img, (x1, y1), (x2, y2), (0, 0, 255), 2)

        (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(original_img, (x1, y1 - h - 5), (x1 + w, y1), (0, 0, 255), -1)
        cv2.putText(original_img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

      original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
      plt.figure(figsize=(10, 10))
      plt.imshow(original_img)
      plt.show()

    match = re.search(r"(\d+)_(\d+)_(\d+)\.txt", img_name)
    if match:
      row, col = int(match.group(1)), int(match.group(2))
    else:
      raise ValueError(f"Filename format incorrect: {img_name}")

    if row + 1 == num_row:
      offset_y = height - block_size
    else:
      offset_y = row * stride

    if col + 1 == num_col:
      offset_x = width - block_size
    else:
      offset_x = col * stride

    with open(predict_path, "r") as f:
      labels = [line.strip() for line in f]

    for label_str in labels:

      label = label_str.split(" ")

      id = int(label[0])
      x = int(float(label[1]) * block_size)
      y = int(float(label[2]) * block_size)
      w = int(float(label[3]) * block_size)
      h = int(float(label[4]) * block_size)
      conf = float(label[5])

      x1 = x - w // 2
      y1 = y - h // 2
      x2 = x + w // 2
      y2 = y + h // 2

      x1 = x1 + offset_x
      y1 = y1 + offset_y
      x2 = x2 + offset_x
      y2 = y2 + offset_y

      detection.append((id, x1, y1, x2, y2, conf))

merged_boxes = merge_bounding_box(detection)

#繪製邊界框跟label(紅底白字)
for class_id, x1, y1, x2, y2, confidence in merged_boxes:
  label = f"{class_name[class_id]} {confidence:.2f}"
  # print(f"label:{label}, location:{(x1, y1), (x2, y2)}")
  cv2.rectangle(original_img, (x1, y1), (x2, y2), (0, 0, 255), 2)

  (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
  cv2.rectangle(original_img, (x1, y1 - h - 5), (x1 + w, y1), (0, 0, 255), -1)
  cv2.putText(original_img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 10))
plt.imshow(original_img)
plt.show()